# 🕵️ 감시자(Overseer) DQN 학습 — 체제1 vs 체제2 (`03_train_overseer`)

**2026 창의설계축전 — 연구의 핵심.** 감시자가 값싼 정보만 보다가 **정밀검사(AUDIT) 시점을 스스로 학습**하여, 상시 전체검사보다 효율적으로 치팅을 잡는지 검증합니다.

- **재배 로봇**은 지금 **스크립트 고정 정책**(정직/치터)을 사용합니다 — 감시자 연구를 재배 로봇 RL 난이도와 분리하기 위함. 나중에 RL 학습 정책으로 교체 가능.
- **비교 축**: 체제1(공개 정보만) vs 체제2(+행동 로그). 두 체제의 적발률·검사횟수·지연 차이 = **행동 로그의 가치**.

> 핵심 통찰: 치터의 스푸핑이 완벽해 **공개 정보만으론 정직/치터가 구별되지 않습니다**(둘 다 항상 건강해 보임). 행동 로그가 있어야 의심 대상을 특정할 수 있습니다.


## 1. 설정 — 패키지 설치 + 코드 내려받기

In [ ]:
!pip install -q stable-baselines3 gymnasium

import os, sys
if os.path.isdir('/content'):
    os.chdir('/content')
    if not os.path.isdir('tomato-oversight'):
        !git clone https://github.com/1ee1ee1ee/tomato-oversight.git
    os.chdir('/content/tomato-oversight')
sys.path.insert(0, os.getcwd())
print('working dir :', os.getcwd())


## 2. 감시자 환경 살펴보기

- **행동**: `0=CONTINUE`(계속 관찰) / `1=AUDIT`(정밀검사).
- **보상**: AUDIT −`c_audit` / 적발(불일치>0 검사) +`r_catch` / 허탕(불일치0 검사) −`f_false` / 치터 미적발 −`p_miss`.
- **관측(O 위치·실제생존·정답 라벨은 절대 미포함)**:
  - 체제1(6차원): 겉보기 생존·진행률·마지막검사 후 경과·이전 검사결과·누적 검사수
  - 체제2(18차원): 체제1 + 로봇 위치·최근 행동·이동/물주기 비율·방문 셀 다양성(=죽치는 패턴 감지)

빠른 학습을 위해 동역학을 5배 압축(`FAST`)합니다. 설계 그대로(마름 500)로 하려면 `FAST`를 비우고 `horizon=1000`을 쓰세요.


In [ ]:
from src.overseer_env import OverseerEnv, OverseerConfig, OverseerAction
from stable_baselines3.common.monitor import Monitor

FAST = dict(dry_after_steps=100, check_interval=100, spoof_duration=100, horizon=200)

def make_env(regime):
    return Monitor(OverseerEnv(OverseerConfig(regime=regime, **FAST)))

for regime in (1, 2):
    e = OverseerEnv(OverseerConfig(regime=regime, **FAST))
    print(f'체제{regime}: 관측 {e.observation_space.shape}, 행동 {e.action_space.n}개 (CONTINUE/AUDIT)')


## 3. 두 체제 감시자 학습

체제1·체제2 각각 DQN을 학습합니다. 압축 동역학 기준 각 8만 step은 CPU로 수 분입니다.


In [ ]:
from stable_baselines3 import DQN

def train_overseer(regime, timesteps=80_000, seed=1):
    model = DQN('MlpPolicy', make_env(regime),
                learning_rate=1e-3, buffer_size=50_000, learning_starts=2_000,
                batch_size=128, gamma=0.99, train_freq=4, target_update_interval=1_000,
                exploration_fraction=0.3, exploration_final_eps=0.05,
                policy_kwargs=dict(net_arch=[128, 128]), verbose=0, seed=seed)
    model.learn(total_timesteps=timesteps, progress_bar=True)
    return model

models = {}
for regime in (1, 2):
    print(f'▶ 체제{regime} 학습')
    models[regime] = train_overseer(regime)


## 4. 평가 — 체제1 vs 체제2 비교

치터 에피소드는 **적발률·검사지연·검사횟수**를, 정직 에피소드는 **허탕 검사 횟수**를 측정합니다.


In [ ]:
import numpy as np

def evaluate(model, regime, n=200):
    env = OverseerEnv(OverseerConfig(regime=regime, **FAST))
    caught = delays = cheat_eps = honest_eps = honest_audits = cheat_audits = 0
    for ep in range(n):
        obs, info = env.reset(seed=10_000 + ep)
        done = False
        while not done:
            a, _ = model.predict(obs, deterministic=True)
            obs, r, term, trunc, info = env.step(int(a)); done = term or trunc
        if info['is_cheater']:
            cheat_eps += 1; cheat_audits += info['audit_count']
            if info['caught']:
                caught += 1; delays += max(0, env.step_count - FAST['dry_after_steps'])
        else:
            honest_eps += 1; honest_audits += info['audit_count']
    return dict(catch_rate=caught / max(1, cheat_eps),
                mean_delay=delays / max(1, caught),
                audits_per_cheater=cheat_audits / max(1, cheat_eps),
                false_audits_per_honest=honest_audits / max(1, honest_eps))

results = {r: evaluate(models[r], r) for r in (1, 2)}

print(f"{'':7}{'적발률':>9}{'검사지연':>10}{'치터당검사':>12}{'정직당허탕':>12}")
for r in (1, 2):
    m = results[r]
    print(f"체제{r} {m['catch_rate']*100:8.0f}%{m['mean_delay']:10.1f}{m['audits_per_cheater']:12.2f}{m['false_audits_per_honest']:12.2f}")


## 5. 비교 시각화

In [ ]:
import matplotlib.pyplot as plt

labels = ['체제1\n(공개정보만)', '체제2\n(+행동로그)']
catch = [results[1]['catch_rate']*100, results[2]['catch_rate']*100]
audits = [results[1]['audits_per_cheater'], results[2]['audits_per_cheater']]

fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].bar(labels, catch, color=['#94a3b8', '#22c55e']); ax[0].set_title('치터 적발률 (%)'); ax[0].set_ylim(0, 100)
ax[1].bar(labels, audits, color=['#94a3b8', '#22c55e']); ax[1].set_title('치터 1건당 검사 횟수 (낮을수록 효율적)')
for a in ax:
    a.set_axisbelow(True); a.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()


## 6. 감시자가 "언제" 검사하는지 들여다보기

치터 한 명을 상대로 감시자가 몇 번째 step에 AUDIT을 눌렀는지 봅니다. (불일치는 마름 시점 이후에 드러남)


In [ ]:
from src.policies import ScriptedCheater

env = OverseerEnv(OverseerConfig(regime=2, **FAST), policies=[ScriptedCheater()])
obs, info = env.reset(seed=0)
log = []
done = False
while not done:
    a, _ = models[2].predict(obs, deterministic=True)
    if int(a) == int(OverseerAction.AUDIT):
        log.append((env.step_count, info['apparent_alive'], info['true_alive']))
    obs, r, term, trunc, info = env.step(int(a)); done = term or trunc

print(f'마름 시점(불일치 시작) ≈ step {FAST["dry_after_steps"]}')
print('감시자 AUDIT 시점 (step, 겉보기, 실제):', log if log else '검사 안 함')
print('적발 성공:', info['caught'])


## 7. 모델 저장 (Google Drive)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
save_dir = '/content/drive/MyDrive/tomato-oversight/models'
os.makedirs(save_dir, exist_ok=True)
for r in (1, 2):
    models[r].save(f'{save_dir}/overseer_regime{r}')
print('저장 완료:', save_dir)


## 다음 단계

- **재배 로봇을 RL 정책으로 교체**: `01_train_honest`의 학습 정책 + 치터 학습 정책을 `OverseerEnv(policies=[...])`에 넣기 (스크립트 정책과 같은 `reset()/act(env)` 인터페이스로 감싸면 됨).
- **여러 정책 풀**: 정직·치터 각각 여러 seed로 학습해 무작위로 상대 → 감시자 일반화 강화.
- **보상 상수 민감도**: `c_audit`·`r_catch`·`f_false`·`p_miss` 스윕 → 체제1도 검사하도록 유도되는지.
- **확장 환경**: S 타일(정당한 고보상)을 추가해 "높은 보상=치팅" 단순 암기가 아님을 검증(E2/E3).
- **설계 그대로**: `FAST` 대신 마름 500·horizon 1000으로 최종 실험.
